In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains

import time

In [59]:
driver = webdriver.Chrome()
driver.maximize_window()

# Explicit wait
wait = WebDriverWait(driver,20)

# A function to check web page is fully loaded
def wait_for_page_to_load(driver,wait):
    page_title = driver.title
    # wait for webpage to load
    try:
        wait.until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

    except:
        print(f"The page \"{page_title}\" did not get fully loaded within the given duration.\n")
    
    else:
        print(f"The page \"{page_title}\" is successfully loaded!\n")
        


url = "https://finance.yahoo.com/"
driver.get(url)
wait_for_page_to_load(driver,wait)

# Declining cookies
try:
    cookie_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="consent-page"]/div/div/div/form/div[2]/div[2]/button[2]')
        )
    )
    cookie_btn.click()

except:
    print("Cookie popup not found.")

actions = ActionChains(driver)

# Hovering on Markets menu
markets_menu = wait.until(EC.presence_of_element_located((By.XPATH,"//div[normalize-space()='Markets']")))
actions.move_to_element(markets_menu).perform()

# Hovering to Stocks inside Market menu
stocks_menu = wait.until(EC.visibility_of_element_located((By.XPATH,'/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/a[1]')))
actions.move_to_element(stocks_menu).perform()

# Hovering to -> Trending Tickers
trending_tickers = wait.until(EC.visibility_of_element_located((By.XPATH,'/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[4]/a[1]/span[1]')))
actions.move_to_element(trending_tickers).perform()

# Clicking Trending Tickers
trending_tickers_click = wait.until(EC.element_to_be_clickable((By.XPATH, '/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[4]/a[1]')))
trending_tickers_click.click()

wait_for_page_to_load(driver,wait)

# Click on Most Active
most_active = wait.until(EC.element_to_be_clickable((By.XPATH,'/html[1]/body[1]/div[1]/div[4]/main[1]/section[1]/section[1]/section[1]/section[1]/section[1]/div[1]/div[1]/div[1]/a[1]')))
most_active.click()
wait_for_page_to_load(driver,wait)

# scraping the data
data = []
while True:
    # scraping
    wait.until(EC.presence_of_element_located((By.TAG_NAME,"table")))
    rows = driver.find_elements(By.CSS_SELECTOR,"table tbody tr")

    for row in rows:
        values = row.find_elements(By.TAG_NAME,"td")
        stock = {
            "name": values[1].text,
			"symbol": values[0].text,
			"price": values[3].text,
			"change": values[4].text,
			"volume": values[6].text,
			"market_cap": values[8].text,
			"pe_ratio": values[9].text,
        }
        data.append(stock)
        
    

    # click next
    try:
        next_button = wait.until(EC.element_to_be_clickable((By.XPATH,'/html/body/div[1]/div[4]/main/section/section/section/section/section[1]/div/div[4]/div[3]/button[3]')))
    except:
        print("The \"next button\" is not clickable. We have navigated through all the pages.")
        break
    else:
        next_button.click()
        time.sleep(1)




driver.quit()


The page "Ihre Datenschutzeinstellungen" is successfully loaded!

The page "Yahoo Finance - Stock Market Live, Quotes, Business & Finance News" is successfully loaded!

The page "Top Trending Stocks: US stocks with the highest interest today - Yahoo Finance" is successfully loaded!

The "next button" is not clickable. We have navigated through all the pages.


In [65]:
import numpy as np 
import pandas as pd 

In [80]:
stocks_df = (
    pd
    .DataFrame(data)

    # Strip spaces from text columns
    .apply(
        lambda col: col.str.strip()
        if col.dtype == "object"
        else col
    )

    .assign(

        # Price
        price=lambda df_:
            pd.to_numeric(
                df_.price,
                errors="coerce"
            ),

        # Change
        change=lambda df_:
            pd.to_numeric(
                df_.change
                    .replace(["-", "--", "N/A"], np.nan)
                    .str.replace("+", "", regex=False),
                errors="coerce"
            ),

        # Volume (assumes M values)
        volume=lambda df_:
            pd.to_numeric(
                df_.volume
                    .replace(["-", "--", "N/A"], np.nan)
                    .str.replace("M", "", regex=False),
                errors="coerce"
            ),

        # Market cap -> convert all to Billions
        market_cap=lambda df_:
            df_.market_cap.apply(
                lambda val:
                    np.nan if pd.isna(val) or val in ["-", "--", "N/A"] else
                    float(val.replace("M", ""))/1000 if "M" in val else
                    float(val.replace("B", "")) if "B" in val else
                    float(val.replace("T", ""))*1000 if "T" in val else
                    np.nan
            ),

        # P/E ratio
        pe_ratio=lambda df_:
            pd.to_numeric(
                df_.pe_ratio
                    .replace(["-", "--", "N/A"], np.nan)
                    .str.replace(",", "", regex=False),
                errors="coerce"
            )
    )

    .rename(columns={
        "price": "price_usd",
        "volume": "volume_M",
        "market_cap": "market_cap_B"
    })
)

stocks_df

,name,symbol,price_usd,change,volume_M,market_cap_B,pe_ratio
0,Faraday Future Intelligent Electric Inc.,FFAI,0.5319,0.2452,563.673,0.132321,NaN
1,UnitedHealth Group Incorporated,UNH,346.0100,22.5300,23.537,314.065000,26.15
2,Opendoor Technologies Inc.,OPEN,5.4500,0.1000,75.686,5.223000,NaN
3,Plug Power Inc.,PLUG,3.0800,-0.1400,79.186,4.293000,NaN
4,Ondas Inc.,ONDS,10.8700,0.1400,79.012,5.243000,NaN
...,...,...,...,...,...,...,...
310,Synchrony Financial,SYF,77.6300,-0.9500,5.087,27.960000,8.44
311,The Kroger Co.,KR,68.8100,0.9200,5.048,43.546000,44.28
312,"Apollo Global Management, Inc.",APO,127.2600,-0.0700,5.025,73.865000,22.51
313,"Flagstar Bank, National Association",FLG,14.4100,-0.4000,5.019,5.994000,NaN
